In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [2]:
import torch

from src.federated.client import FederatedClient
from src.federated.server import FederatedServer

from src.preprocessing.dataset_loader import build_dataset_index
from src.preprocessing.splitter import create_stratified_split
from src.preprocessing.dataloaders import (
    create_datasets,
    create_dataloaders
)

In [3]:
train_paths, train_labels = build_dataset_index(
    "../data/raw/train"
)

test_paths, test_labels = build_dataset_index(
    "../data/raw/test"
)

(
    X_train,
    X_val,
    y_train,
    y_val
) = create_stratified_split(
    train_paths,
    train_labels
)

(
    train_dataset,
    val_dataset,
    test_dataset
) = create_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    test_paths,
    test_labels
)

(
    train_loader,
    val_loader,
    test_loader
) = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32
)

In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [5]:
client1 = FederatedClient(
    "client_1",
    train_loader,
    device
)

client2 = FederatedClient(
    "client_2",
    train_loader,
    device
)

client3 = FederatedClient(
    "client_3",
    train_loader,
    device
)

clients = [
    client1,
    client2,
    client3
]

In [6]:
server = FederatedServer(device)

server.distribute_weights(clients)

In [7]:
print(
    torch.equal(
        client1.get_weights()["classifier.2.weight"],
        client2.get_weights()["classifier.2.weight"]
    )
)

True


In [8]:
print("\nTraining clients...\n")

client1.train(epochs=1)
client2.train(epochs=1)
client3.train(epochs=1)


Training clients...

0.0008194178226403892 0.9991023540496826
0.027245795354247093 0.9915002584457397
1.908380181703251e-05 0.9999014139175415
0.03130878135561943 0.9700779318809509
3.4564636735012755e-05 0.9999054670333862
0.03575831279158592 0.9984367489814758
9.714828047435731e-05 0.9998800754547119
0.0666499137878418 0.9935234785079956
2.3171603970695287e-05 0.9999769926071167
0.08711960911750793 0.996423065662384
0.00020278365991543978 0.9997237324714661
0.06718585640192032 0.9964757561683655
9.774923819350079e-05 0.9998608827590942
0.134739488363266 0.9973759651184082
0.0003252832102589309 0.9995033740997314
0.11821845173835754 0.9955148100852966
0.0004404303035698831 0.9993801116943359
0.15338636934757233 0.9951179027557373
0.00033423874992877245 0.9994753003120422
0.1617029756307602 0.9964978694915771
0.0005848730797879398 0.9991724491119385
0.12501385807991028 0.9969881176948547
0.0005978443077765405 0.9993888139724731
0.13144850730895996 0.9970508813858032
0.0011158298002555

(0.3427688152962969, 0.8497137404580153)

In [9]:
print(
    torch.equal(
        client1.get_weights()["classifier.2.weight"],
        client2.get_weights()["classifier.2.weight"]
    )
)

False


In [10]:
server.aggregate(
    clients
)

In [11]:
server.distribute_weights(
    clients
)

In [12]:
print(
    torch.equal(
        client1.get_weights()["classifier.2.weight"],
        client2.get_weights()["classifier.2.weight"]
    )
)

True


In [13]:
same = True

w1 = client1.get_weights()
w2 = client2.get_weights()

for key in w1:

    if not torch.equal(
        w1[key],
        w2[key]
    ):
        same = False
        break

print(same)

True
